# Clustering Analysis
The pipeline pre-processes data and extends the DAO object. To run the pipeline you must pass the path to the raw data location. To download the data see the following link.

https://www.kaggle.com/c/m5-forecasting-accuracy/data

In [ ]:
# the notebooks import src/ and read data/ relative to the project folder
import os
if os.path.basename(os.getcwd()) == 'code':
    os.chdir('..')

In [2]:
# Import the pipeline class
from src.preprocessing.Pipeline import Pipeline

# import package to help analize clustering
from src.clustering.ClusterAnalysis import ClusterAnalysis

# import KMeans
from sklearn.cluster import KMeans

# import plotly
import plotly.express as px

# Setup Pipeline and Build Features

In [3]:
# setup pipeline
pipeline = Pipeline(path='data/raw/',verbose = False)

# get feature set
df = pipeline.buildFeatureSet()

Encoding ['item_id']


# Data-Preprocessing
remove nan's and setup the feature set for the k-means clustering.

In [5]:
# columns to remove
remove = ['id', 'item_id', 'dept_id', 'store_id', 'cat_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk']

# columns to be used as features
features = [ col for col in df.columns if col not in remove]

# remove nans
df = df.dropna()

# sample 1M rows
df = df.sample(1000000)

# Perform Clustering Algorithm

In [ ]:
# setup clustering analysis object
analysis = ClusterAnalysis()

# create clusters clusters between two and 5
for cluster in range(2,15):
    
    # create kmeans object
    kmeans = KMeans(n_clusters=cluster,max_iter=2000,tol=0.00001)

    # fit kmeans object to data
    kmeans.fit(df[features].values)
    
    # add record to dataset
    analysis.addRecord(kmeans,df[features].values,cluster)
    
# display cluster results
analysis.display()

In [ ]:
scores = analysis.scores.copy()
scores['score'] = scores.score/1000000
scores

# Creake K Means Cluster
cluster = 4

In [ ]:
# create kmeans object
kmeans = KMeans(n_clusters=4,max_iter=2000,tol=0.00001)

# fit kmeans object to data
kmeans.fit(df[features].values)

# save new clusters for chart
df['Cluster_k_means'] = kmeans.fit_predict(df[features].values)

# Perform EDA Anaysis

In [ ]:
# create plot data frame
df_plot = df.groupby('Cluster_k_means')[['sales','item_nunique_price']].mean().reset_index()

df_plot = df_plot.rename(columns={'item_nunique_price':'Unique Items','Cluster_k_means':'Cluster','sales':'Sales'})

df_plot['Cluster'] = df_plot['Cluster'].apply(lambda x: 'Custer '+ str(x))

fig = px.bar(df_plot, x='Cluster', y='Sales',color='Unique Items')

# update figure layout
fig.update_layout(autosize=False, width=600, height=600)

# show figure
fig.show()

In [ ]:
key = 'item_nunique_price'

df_plot = df.groupby('Cluster_k_means')[[key]].mean().reset_index()
df_plot[key] = df_plot[key] 

fig = px.bar(df_plot, x='Cluster_k_means', y=key)
fig.show()

In [ ]:
features

In [ ]:
# columns to remove
remove = ['Cluster', 'Cluster_k_means','id', 'item_id', 'dept_id', 'store_id', 'cat_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk']

# columns to be used as features
features = [ col for col in df.columns if col not in remove]


# setup clustering analysis object
analysis3 = ClusterAnalysis()

# create clusters clusters between two and 5
for cluster in range(2,15):
    
    # create kmeans object
    clustering = AgglomerativeClustering(n_clusters = cluster).fit(df[features].values)
        
    # add record to dataset
    analysis3.addRecord(kmeans,df[features].values,cluster)
    
# display cluster results
analysis3.display()

# Save Data
Here the data is saved to perform further classification analysis in R.

In [ ]:
# save as clustering data
df.to_csv('data/refined/clustering_data.csv')